# Complexity-03b — Le théorème de hiérarchie temporelle de Hartmanis et Stearns (1965)

**Approfondissement de [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-CSharp.ipynb)** | [Version Python](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) | [Index de la série](README.md)

> **Environnement** : C# et .NET 9 ou ultérieur ; noyau .NET Interactive. Le code de ce jumeau n'utilise que la BCL : aucune dépendance externe.

**Ce que ce notebook suppose en plus de sa base** : la diagonalisation sous budget de 03, puis la lecture de machines de Turing multitêtes et d'une borne de simulation. Les sections 1 et 2 reprennent des expériences déjà accessibles dans 01 et 02 ; les sections 3 et 4 étudient le théorème historique. Cette accrétion de niveau Recherche n'est pas nécessaire au parcours des numéros nus.

> **Hommage.** Richard E. Stearns (5 juillet 1936 — 29 août 2026, Ann Arbor) a reçu avec Juris
> Hartmanis le **prix Turing 1993** *« in recognition of their seminal paper which established the
> foundations for the field of computational complexity theory »*. Leur article de 1965 — écrit
> dans un laboratoire industriel de GE à Schenectady, pour un champ qui n'avait pas encore de nom —
> **lui a donné son nom** : *On the computational complexity of algorithms*, Trans. Amer. Math.
> Soc. 117, 285–306. Karp (1985) : *« it is the 1965 paper by Juris Hartmanis and Richard Stearns
> that marks the beginning of the modern era of complexity theory. »* Ce notebook travaille leur
> théorème. Sources biographiques : [CACM In Memoriam](https://cacm.acm.org/news/in-memoriam-richard-e-stearns-1936-2026/)
> (Spafford & Garfinkel, 03/09/2026) et [Computational Complexity blog](https://blog.computationalcomplexity.org/2026/09/richard-stearns-1936-2026.html)
> (Gasarch, 04/09/2026).

> **Écho dans le dépôt.** Ce notebook a été la **pierre inaugurale de la série `Complexity/`** — le
> geste fondateur de Hartmanis et Stearns (compter les pas d'une machine comme on compte des
> mètres) manquait au corpus, alors que les séries voisines l'effleurent :
> [SL-10 — Apprentissage actif d'automates](../../SymbolicAI/SymbolicLearning/SL-10-ActiveAutomataLearning-Python.ipynb)
> (automates, sans le théorème qui en fonde la hiérarchie),
> [App-13 — TSP métaheuristiques](../../Search/Part3-Cas-pratiques/App-13-TSP-Metaheuristics-CSharp.ipynb)
> (Stearns a produit la première analyse de pire cas des heuristiques du voyageur de commerce),
> et [SocialChoice-01 — Arrow](../../GameTheory/SocialChoice/ChoixSocial-01-Arrow-Impossibility-Theorem-Python.ipynb)
> (le **tout premier papier** de Stearns, étudiant, portait sur le paradoxe d'Arrow — 1959).
> Précédent d'hommage par l'œuvre : [Coalitions-04 — Munkres](../../GameTheory/Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb).


## 0. Pourquoi ce notebook

Le geste de 1965 tient en une phrase : **le temps est une ressource mesurable, et le mesurer
crée une hiérarchie stricte de problèmes**. Avant Hartmanis et Stearns, on savait qu'un
problème était « décidable » ou non ; après eux, on sait demander *en combien de pas*. La
rétrospective de Stearns sur Shannon donne la motivation d'origine : *« he thought there must
be a reason why things are hard »* — il doit y avoir une raison pour que certaines choses
soient dures, et cette raison se mesure.

Ce notebook suit le papier en quatre gestes, **tous exécutés** dans des cellules C#. Les
deux premiers recoupent les notebooks d'entrée de la série
([01 — Compter des pas](Complexity-01-StepCounting-CSharp.ipynb),
[02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-CSharp.ipynb)), qui les reprennent pas à
pas ; les deux suivants sont propres à ce notebook.

1. **Compter les pas** — un simulateur de machine de Turing multitête (le modèle exact du
   papier de 1965), avec compteur de pas : la hiérarchie se lit dans des comptes exacts, pas
   dans des chronomètres (section 1) ;
2. **Séparer empiriquement** — là où le chronomètre parle : force brute contre programmation
   dynamique sur subset-sum, le point de bascule mesuré (section 2) ;
3. **Diagonaliser** — l'argument de Cantor–Turing adapté aux budgets de temps, construit et
   vérifié sur une famille finie de machines (section 3) ;
4. **Énoncer le théorème** — la hiérarchie de 1965, ses hypothèses exactes
   (temps-constructibilité), et le prix de la simulation rendu visible (section 4).

La section 5 relie l'homme à l'œuvre inattendue (Arrow, jeux répétés), la section 6 mesure
honnêtement ce qui existe et manque dans Mathlib pour formaliser ce théorème, la section 7
conclut.


In [1]:
// Outillage commun : tout le carnet n'utilise que la bibliotheque standard .NET.
#nullable enable
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using System.Numerics;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.GetCultureInfo("fr-FR");
Console.WriteLine("Complexity-03b — Hartmanis-Stearns (1965)");
Console.WriteLine($"Runtime observé : .NET {Environment.Version}; algorithmes écrits from-scratch, BCL uniquement.");


Complexity-03b — Hartmanis-Stearns (1965)


Runtime observé : .NET 10.0.13; algorithmes écrits from-scratch, BCL uniquement.


## 1. Le geste fondateur : compter les pas

Le papier de 1965 ne chronomètre pas : il **compte**. Le modèle est la machine de Turing
**multitête** (plusieurs rubans, une tête par ruban), et la ressource est le **nombre de pas**
exact de la machine. Ce compte est déterministe, reproductible, indépendant du matériel — c'est
l'unité de mesure qui fonde la discipline.

Le simulateur ci-dessous implémente ce modèle en C# : des rubans infinis aux deux extrémités, une
tête par ruban, des transitions $(q, a_1, \dots, a_k) \mapsto (q', \text{écritures}, \text{déplacements})$,
et surtout un **compteur de pas** — l'instrument du geste fondateur. Une machine s'arrête soit
dans un **état acceptant**, soit par blocage (aucune transition applicable) : le rejet est le
blocage, comme sur les machines du papier.


In [2]:
#nullable enable
class Ruban
{
    public Dictionary<int,char> Cellules { get; } = new();
    public int Tete { get; set; }
    public Ruban(string mot = "") { for (int i=0;i<mot.Length;i++) Cellules[i]=mot[i]; }
    public char Lire() => Cellules.TryGetValue(Tete,out char c) ? c : '_';
    public void Ecrire(char c) => Cellules[Tete]=c;
    public void Deplacer(char d) => Tete += d=='R' ? 1 : d=='L' ? -1 : 0;
}

record ActionRuban(char Ecrire, char Direction);
record RegleTuring(string NouvelEtat, ActionRuban[] Actions);

class MachineTuring
{
    public int K { get; }
    public Dictionary<string,RegleTuring> Delta { get; }
    public string Initial { get; }
    public HashSet<string> Acceptants { get; }
    public long Pas { get; private set; }
    public MachineTuring(int k, Dictionary<string,RegleTuring> delta, string initial, IEnumerable<string> acceptants)
    { K=k; Delta=delta; Initial=initial; Acceptants=new HashSet<string>(acceptants); }
    public (string? Etat, List<Ruban> Rubans) Run(string mot, long budget=100_000)
    {
        var rubans=Enumerable.Range(0,K).Select(i=>new Ruban(i==0?mot:"")).ToList();
        string etat=Initial; Pas=0;
        while(!Acceptants.Contains(etat))
        {
            if(Pas>=budget) return (null,rubans);
            string cle=Cle(etat,rubans.Select(r=>r.Lire()));
            if(!Delta.TryGetValue(cle,out var regle)) return (null,rubans);
            etat=regle.NouvelEtat;
            for(int i=0;i<K;i++){rubans[i].Ecrire(regle.Actions[i].Ecrire);rubans[i].Deplacer(regle.Actions[i].Direction);}
            Pas++;
        }
        return (etat,rubans);
    }
    public static string Cle(string etat,IEnumerable<char> symboles) => etat+"|"+string.Join("|",symboles);
}

string LireRuban(Ruban r)
{
    if(r.Cellules.Count==0) return "";
    int fin=r.Cellules.Keys.Max();
    return new string(Enumerable.Range(0,fin+1).Select(i=>r.Cellules.TryGetValue(i,out char c)?c:'_').ToArray()).TrimEnd('_');
}

// --- Machine A : increment binaire, 1 ruban ---
var transitionsIncrement = new Dictionary<string,RegleTuring>();
void AjouteIncrement(string etat,string lu,string nouvel,char ecrit,char dir) => transitionsIncrement[MachineTuring.Cle(etat,lu)] = new RegleTuring(nouvel,new[]{new ActionRuban(ecrit,dir)});
AjouteIncrement("q0","0","q0",'0','R'); AjouteIncrement("q0","1","q0",'1','R'); AjouteIncrement("q0","_","q1",'_','L');
AjouteIncrement("q1","1","q1",'0','L'); AjouteIncrement("q1","0","qF",'1','S'); AjouteIncrement("q1","_","qF",'1','S');
var inc = new MachineTuring(1,transitionsIncrement,"q0",new[]{"qF"});

var (etatInc,rubansInc) = inc.Run("1011");
string luInc = LireRuban(rubansInc[0]);
if(luInc!="1100" || etatInc!="qF" || inc.Pas!=8) throw new Exception($"Increment binaire incorrect : {luInc}, etat={etatInc}, {inc.Pas} pas.");
Console.WriteLine($"increment('1011') -> {luInc} (attendu 1100), etat={etatInc}, {inc.Pas} pas");


increment('1011') -> 1100 (attendu 1100), etat=qF, 8 pas


In [3]:
#nullable enable
// --- Machine B : palindrome sur {0,1}, 1 ruban, zigzag O(n^2) ---
// Comparer premier et dernier caractere, les rayer ('x'), recommencer.
// Rejet = etat puits 'rejeter' sans transition sortante -> blocage.
var transitionsPal = new Dictionary<string,RegleTuring>();
void AjoutePal(string etat,string lu,string nouvel,ActionRuban action) => transitionsPal[MachineTuring.Cle(etat,lu)] = new RegleTuring(nouvel,new[]{action});
AjoutePal("scan","0","marque0",new ActionRuban('x','R')); AjoutePal("scan","1","marque1",new ActionRuban('x','R')); AjoutePal("scan","x","scan",new ActionRuban('x','R')); AjoutePal("scan","_","fin",new ActionRuban('_','S'));
foreach(string s in new[]{"0","1"}) { AjoutePal("marque0",s,"marque0",new ActionRuban(s[0],'R')); AjoutePal("marque1",s,"marque1",new ActionRuban(s[0],'R')); }
AjoutePal("marque0","x","verif0",new ActionRuban('x','L')); AjoutePal("marque0","_","verif0",new ActionRuban('_','L')); AjoutePal("marque1","x","verif1",new ActionRuban('x','L')); AjoutePal("marque1","_","verif1",new ActionRuban('_','L'));
AjoutePal("verif0","0","retour",new ActionRuban('x','L')); AjoutePal("verif1","1","retour",new ActionRuban('x','L'));
AjoutePal("verif0","1","rejeter",new ActionRuban('1','S')); AjoutePal("verif1","0","rejeter",new ActionRuban('0','S'));
AjoutePal("verif0","x","fin",new ActionRuban('x','S')); AjoutePal("verif1","x","fin",new ActionRuban('x','S'));
foreach(string s in new[]{"0","1","x"}) AjoutePal("retour",s,"retour",new ActionRuban(s[0],'L'));
AjoutePal("retour","_","scan",new ActionRuban('_','R'));
var pal = new MachineTuring(1,transitionsPal,"scan",new[]{"fin"});

// Verification de correction avant toute mesure
foreach(var (mot,attendu) in new[]{("0110","fin"),("0100",(string?)null),("0","fin"),("0110110","fin")})
{
    var (etat,_) = pal.Run(mot);
    if(etat!=attendu) throw new Exception($"pal({mot}) -> {etat}, attendu {attendu}");
}
Console.WriteLine("verifications : pal accepte 0110, 0110110, 0 ; rejette 0100 (blocage). OK");

Console.WriteLine();
Console.WriteLine($"{"n",4} | {"pas inc",8} | {"pas pal",8} | rapport pal/inc");
Console.WriteLine(new string('-',40));
foreach(int demi in new[]{4,8,16,32,64})
{
    string moitie=new string(Enumerable.Range(0,demi).Select(i=>(i*7)%2==0?'0':'1').ToArray());
    string mot=moitie+new string(moitie.Reverse().ToArray());
    inc.Run(mot);
    pal.Run(mot,500_000);
    Console.WriteLine($"{mot.Length,4} | {inc.Pas,8} | {pal.Pas,8} | {(double)pal.Pas/inc.Pas,12:F2}x");
}

Console.WriteLine();
Console.WriteLine("Le compte exact confirme la theorie : increment ~ lineaire, palindrome ~ quadratique.");
Console.WriteLine("C'est LE geste de 1965 : la ressource n'est pas un chrono, c'est un compte de pas.");


verifications : pal accepte 0110, 0110110, 0 ; rejette 0100 (blocage). OK


   n |  pas inc |  pas pal | rapport pal/inc


----------------------------------------


   8 |       10 |       69 |         6,90x


  16 |       18 |      233 |        12,94x


  32 |       34 |      849 |        24,97x


  64 |       66 |     3233 |        48,98x


 128 |      130 |    12609 |        96,99x


Le compte exact confirme la theorie : increment ~ lineaire, palindrome ~ quadratique.


C'est LE geste de 1965 : la ressource n'est pas un chrono, c'est un compte de pas.


**Lecture.** Les deux machines tournent sur la même famille d'entrées (des palindromes de
longueur croissante), et leurs comptes de pas croissent comme $\Theta(n)$ et $\Theta(n^2)$. La
définition fondatrice s'écrit alors naturellement :

$$\mathrm{TIME}(f) = \{ L \mid \text{une machine multitête décide } L \text{ en } O(f(n)) \text{ pas} \}$$

Tout l'édifice de la discipline est dans cette parenthèse : la **classe** n'est pas une
propriété d'un problème isolé mais d'une **famille** d'instances mesurée en nombre de pas.
La question fondatrice de Hartmanis et Stearns devient : *ces ensembles sont-ils emboîtés
strictement ?* Y a-t-il des problèmes qui exigent *réellement* plus de pas — pas par
maladresse de programmeur, mais par nature ?


## 2. Séparer : ce que « plus de temps achète » se mesure

Avant le théorème, l'intuition expérimentale. Trois familles d'algorithmes, trois croissances
($n \log n$, $n^2$, $2^n$), mesurées au chronomètre — ici le chrono est légitime : il
approxime le compte de pas à un facteur machine près.

**Subset-sum** servira de séparatrice honnête : le même problème, deux algorithmes —
force brute en $\Theta(2^n \cdot n)$ contre programmation dynamique en $\Theta(n \cdot S)$ —
et l'écart mesuré. C'est la version lisible de ce que le théorème de la section 4 démontre
pour des langages *bien choisis* : plus de budget de temps sépare des classes.


In [4]:
#nullable enable
double Mesurer(System.Action<int> f, int n, int repetitions = 3)
{
    // Temps MINIMAL sur repetitions essais : la course la moins contaminee
    // par l'ordonnanceur (machine partagee) approxime au mieux le cout propre.
    var temps = new List<double>();
    for (int essai = 0; essai < repetitions; essai++)
    {
        var chrono = System.Diagnostics.Stopwatch.StartNew();
        f(n);
        chrono.Stop();
        temps.Add(chrono.Elapsed.TotalSeconds);
    }
    return temps.Min();
}

// --- tri : O(n log n) ---
void Tri(int n)
{
    var rng = new System.Random(0);
    var a = new int[n];
    for (int i = 0; i < n; i++) a[i] = rng.Next(0, 1_000_000);
    System.Array.Sort(a);
    var copie = (int[])a.Clone();
    System.Array.Sort(copie);
    for (int i = 0; i < n; i++) if (a[i] != copie[i]) throw new Exception("tri incorrect");
}

// --- paires : O(n^2) ---
void Paires(int n)
{
    var rng = new System.Random(0);
    var x = new double[n];
    for (int i = 0; i < n; i++) x[i] = rng.NextDouble();
    double somme = 0;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            somme += Math.Abs(x[i] - x[j]);
    if (double.IsNaN(somme)) throw new Exception("paires : somme invalide");
}

// --- boucle 2^n ---
long BoucleExp(int n)
{
    long c = 0;
    long total = 1L << n;
    for (long i = 0; i < total; i++) c += 1;
    return c;
}

if (BoucleExp(10) != (1L << 10)) throw new Exception("compte exact de 2^n incorrect");

int[] taillesTri = new[] { 1_000, 2_000, 4_000, 8_000, 16_000, 32_000 };
int[] taillesPaires = new[] { 250, 500, 1_000, 2_000, 4_000 };
int[] taillesExp = new[] { 18, 19, 20, 21, 22, 23 };

var tTri = new double[taillesTri.Length];
for (int i = 0; i < taillesTri.Length; i++) tTri[i] = Mesurer(n => Tri(n), taillesTri[i]);
var tPaires = new double[taillesPaires.Length];
for (int i = 0; i < taillesPaires.Length; i++) tPaires[i] = Mesurer(n => Paires(n), taillesPaires[i], 3);
var tExp = new double[taillesExp.Length];
for (int i = 0; i < taillesExp.Length; i++) tExp[i] = Mesurer(n => BoucleExp(n), taillesExp[i], 5);

Console.WriteLine("Trois croissances mesurees — l'echelle de la ressource temporelle (minimum de 3 essais) :");
Console.WriteLine($"{"croissance",20} | {"n",7} | {"temps (s)",12}");
Console.WriteLine(new string('-', 44));
for (int i = 0; i < taillesTri.Length; i++) Console.WriteLine($"{"tri n log n",20} | {taillesTri[i],7:N0} | {tTri[i],12:F6}");
for (int i = 0; i < taillesPaires.Length; i++) Console.WriteLine($"{"paires n²",20} | {taillesPaires[i],7:N0} | {tPaires[i],12:F6}");
for (int i = 0; i < taillesExp.Length; i++) Console.WriteLine($"{"boucle 2^n",20} | {taillesExp[i],7:N0} | {tExp[i],12:F6}");

Console.WriteLine();
Console.WriteLine("quand n passe au quadruple (fin de gamme) :");
Console.WriteLine($"  tri    : temps x{tTri[tTri.Length - 1] / tTri[tTri.Length - 3]:F1}");
Console.WriteLine($"  paires : temps x{tPaires[tPaires.Length - 1] / tPaires[tPaires.Length - 3]:F1}");
int n2 = taillesExp[taillesExp.Length - 1] - taillesExp[taillesExp.Length - 3];
Console.WriteLine($"  2^n    : n gagne +{n2} -> operations x{1L << n2} (exact) ; chrono x{tExp[tExp.Length - 1] / tExp[tExp.Length - 3]:F1}");
Console.WriteLine("Le compte d'operations est deterministe ; le chrono le confirme a un facteur machine pres.");


Trois croissances mesurees — l'echelle de la ressource temporelle (minimum de 3 essais) :


          croissance |       n |    temps (s)


--------------------------------------------


         tri n log n |   1 000 |     0,000078


         tri n log n |   2 000 |     0,000216


         tri n log n |   4 000 |     0,000500


... (15 outputs left out)

  tri    : temps x4,4


  paires : temps x18,0


  2^n    : n gagne +2 -> operations x4 (exact) ; chrono x4,6


Le compte d'operations est deterministe ; le chrono le confirme a un facteur machine pres.


In [5]:
#nullable enable
(bool Verdict, long Operations) SubsetBrut(int[] poids, int cible)
{
    // Force brute : enumere les 2^n sous-ensembles.
    // Retourne (verdict, operations) : compte deterministe.
    int n = poids.Length;
    long ops = 0;
    for (int masque = 0; masque < (1 << n); masque++)
    {
        int s = 0;
        for (int i = 0; i < n; i++)
        {
            ops++;
            if ((masque >> i & 1) != 0) s += poids[i];
        }
        if (s == cible) return (true, ops);
    }
    return (false, ops);
}

(bool Verdict, long Cases) SubsetDp(int[] poids, int cible)
{
    // Programmation dynamique : table booleenne sur les sommes 0..cible.
    // Retourne (verdict, cases) : chaque case visitee = 1 operation comptee.
    var atteignable = new bool[cible + 1];
    atteignable[0] = true;
    long cases = 0;
    foreach (int p in poids)
        for (int s = cible; s >= p; s--)   // s >= p : l'index s-p reste toujours dans [0, cible]
        {
            cases++;
            if (atteignable[s - p]) atteignable[s] = true;
        }
    return (atteignable[cible], cases);
}

var rng = new System.Random(42);
Console.WriteLine($"{"n",3} | {"ops brut",10} | {"cases dp",9} | {"brut (s)",9} | {"dp (s)",8}");
Console.WriteLine(new string('-', 56));
foreach (int n in new[] { 8, 11, 14, 17 })
{
    var poids = new int[n];
    for (int i = 0; i < n; i++) poids[i] = rng.Next(1, 60);
    int somme = 0;
    for (int i = 0; i < n; i++) somme += poids[i];
    int cible = somme / 2;
    var chronoBrut = System.Diagnostics.Stopwatch.StartNew();
    var (tb, opsB) = SubsetBrut(poids, cible);
    chronoBrut.Stop();
    var chronoDp = System.Diagnostics.Stopwatch.StartNew();
    var (td, casesD) = SubsetDp(poids, cible);
    chronoDp.Stop();
    if (tb != td) throw new Exception("les deux algorithmes doivent convenir");
    Console.WriteLine($"{n,3} | {opsB,10} | {casesD,9} | {chronoBrut.Elapsed.TotalSeconds,9:F4} | {chronoDp.Elapsed.TotalSeconds,8:F5}");
}

Console.WriteLine();
Console.WriteLine("Meme verdict des deux cotes (verification reussie a chaque ligne). Les compteurs");
Console.WriteLine("d'operations - deterministes, comme les pas comptes en section 1 - remplacent le");
Console.WriteLine("chrono comme preuve : la brute vit sous n * 2^n (arret des qu'un masque convient),");
Console.WriteLine("la dp visite de l'ordre de n * S cases, S = somme visee.");


  n |   ops brut |  cases dp |  brut (s) |   dp (s)


--------------------------------------------------------


  8 |       2048 |       572 |    0,0005 |  0,00030


 11 |       1045 |      1162 |    0,0000 |  0,00001


 14 |       3472 |      2342 |    0,0001 |  0,00001


 17 |      16983 |      3182 |    0,0002 |  0,00002


Meme verdict des deux cotes (verification reussie a chaque ligne). Les compteurs


d'operations - deterministes, comme les pas comptes en section 1 - remplacent le


chrono comme preuve : la brute vit sous n * 2^n (arret des qu'un masque convient),


la dp visite de l'ordre de n * S cases, S = somme visee.


**Lecture.** Les deux algorithmes rendent le même verdict (le contrôle `throw` le prouve à chaque
ligne) mais leurs coûts vivent dans des mondes différents : la force brute double quand $n$
croît de 1, la dynamique reste plate tant que $n \cdot S$ reste modéré. **Aucun des deux n'est
« faux »** — mais l'un vit dans $\mathrm{TIME}(2^{n})$, l'autre dans $\mathrm{TIME}(nS)$.
La question fondatrice se pose alors avec précision : existe-t-il des langages pour lesquels
**aucune machine astucieuse** ne peut descendre sous un certain budget — la dureté est-elle
une propriété du *problème*, pas de l'algorithme qu'on a écrit ? La réponse de 1965 :
**oui, et en quantité dénombrable** — c'est le théorème de hiérarchie, dont l'outil est la
diagonalisation.


### Exercice 1 — Prédire le point de bascule

La DP gagne dès que $2^n$ dépasse $n \cdot S$… en *opérations*. Mais son coût dépend de la
plage des poids : avec des poids dans $[1, 60]$, $S \le 60n$ ; avec des poids dans
$[1, 10^4]$, $S$ explose et la DP peut perdre.

**Objectif** : mesurer le rapport $t_{brut}/t_{dp}$ pour les deux plages de poids
(`[1, 60]` et `[1, 10_000]`), $n$ de 8 à 17, et expliquer en une phrase pourquoi la plage
haute déplace (ou détruit) la bascule.

- **Indice :** reprendre la boucle ci-dessus avec des poids tirés par `rng.Next(1, plage)` sur une graine fixe (`new Random(42)`).
- **Etape 1 :** tracer le rapport t_brut/t_dp pour chaque plage.
- **Etape 2 :** comparer les deux courbes et localiser le croisement (s'il existe).


In [6]:
#nullable enable
// Exercice 1 : a completer
Dictionary<(int plage, int n), double>? resultat_bascule = null; // TODO etudiant : {(plage, n): rapport t_brut/t_dp} mesure
if (resultat_bascule is null)
    Console.WriteLine("Exercice a completer : mesurer le point de bascule pour deux plages de poids");
else
{
    if (resultat_bascule.Count != 20) throw new Exception("Exercice 1 : 20 rapports attendus (2 plages x n de 8 a 17)");
    foreach (var (plage, n) in new[] { (60, 8), (60, 17), (10_000, 8), (10_000, 17) })
        if (!resultat_bascule.ContainsKey((plage, n))) throw new Exception($"Exercice 1 : rapport manquant pour plage={plage}, n={n}");
    Console.WriteLine($"Exercice 1 : 20 rapports fournis ; exemple n=17, plage 60 -> x{resultat_bascule[(60, 17)]:F2}");
}


Exercice a completer : mesurer le point de bascule pour deux plages de poids


## 3. Diagonaliser sous budget : l'argument exécuté

L'argument de Cantor–Turing s'adapte aux budgets de temps. Idée : **énumérer** les machines
$M_0, M_1, M_2, \dots$ et construire une langue $D$ qui **diffère de $M_i$ sur l'entrée $i$** :
simuler $M_i(i)$ pendant un budget $f(i)$, et répondre l'inverse. Si $M_i$ dépasse son budget,
$D$ répond « oui » — et diffère quand même : une machine qui ne tient pas son budget n'est
pas dans $\mathrm{TIME}(f)$, donc ne peut pas être $D$. Aucune machine de la liste ne décide
$D$ **en $f(i)$ pas** ; et si la liste est assez riche, c'est une **preuve d'existence** de
problèmes durs pour chaque budget.

La cellule suivante **exécute** la construction sur une famille finie de cinq machines
(« recognaisseurs » C# sur les indices $0..4$, avec coût en pas explicite) — l'ombre
finie du théorème infini : chaque ligne du tableau montre $D$ différer de $M_i$ **sur sa
propre diagonale**.


In [7]:
#nullable enable
(bool? Verdict, long Pas) Simule(Func<int,(bool verdict, long pas)> machine, int entree, long budget)
{
    // Simule machine(entree) avec un budget de pas.
    // machine retourne (verdict, pas_effectues) ; si pas > budget -> (None, pas).
    var (verdict, pas) = machine(entree);
    if (pas > budget) return (null, pas);
    return (verdict, pas);
}

// Famille finie de "machines" : (verdict sur l'entree i, cout en pas).
var familles = new Func<int,(bool verdict, long pas)>[]
{
    i => (i % 2 == 0, 1 + i),                 // M0 : parite, lineaire
    i => (true, (long)Math.Pow(3, i)),       // M1 : accepte tout, cout exponentiel en i
    i => (i < 3, 3L * i + 2),                // M2 : petit seuil, lineaire
    i => (i % 3 != 0, (long)i * i),          // M3 : mod 3, quadratique
    i => (false, (long)Math.Pow(4, i)),      // M4 : rejette tout, cout exponentiel
};

long BudgetF(int i) => (long)i * i + 10;   // budget diagonal f(i) = i^2 + 10 (joue le role du f log f du theoreme)

Console.WriteLine($"{"i",2} | {"M_i(i)",7} | {"pas",6} | {"budget",7} | {"D(i)",5} | differe");
Console.WriteLine(new string('-', 50));
var D = new Dictionary<int, bool>();
var observes = new Dictionary<int, bool?>();
for (int i = 0; i < familles.Length; i++)
{
    var (verdict, pas) = Simule(familles[i], i, BudgetF(i));
    observes[i] = verdict;
    bool d_i = !(verdict == true);   // rejette si M_i accepte ; accepte sinon (budget inclus)
    D[i] = d_i;
    string affiche = verdict.HasValue ? verdict.Value.ToString() : "None";
    Console.WriteLine($"{i,2} | {affiche,7} | {pas,6} | {BudgetF(i),7} | {d_i,5} | oui");
}

int diffs = Enumerable.Range(0, familles.Length).Count(i => (bool?)D[i] != observes[i]);
Console.WriteLine();
Console.WriteLine($"D differe de chaque M_i sur sa diagonale : {diffs}/{familles.Length} lignes.");
Console.WriteLine("La ligne i=4 exhibe le cas cle : M4(4) coute 4^4 = 256 pas > budget 26 -> verdict");
Console.WriteLine("None, et D(4) = True SANS avoir termine la simulation. C'est legitime : une machine");
Console.WriteLine("qui depasse son budget n'est pas dans TIME(f), donc ne peut pas etre D. Regardez");
Console.WriteLine("aussi M3 : i^2 ne depasse jamais i^2+10 — c'est la MARGE du budget qui la contient ;");
Console.WriteLine("l'Exercice 2 la retire.");
if (diffs != familles.Length) throw new Exception($"la diagonale doit differer sur les {familles.Length} lignes");


 i |  M_i(i) |    pas |  budget |  D(i) | differe


--------------------------------------------------


 0 |    True |      1 |      10 | False | oui


 1 |    True |      3 |      11 | False | oui


 2 |    True |      8 |      14 | False | oui


 3 |   False |      9 |      19 |  True | oui


... (3 outputs left out)

None, et D(4) = True SANS avoir termine la simulation. C'est legitime : une machine


qui depasse son budget n'est pas dans TIME(f), donc ne peut pas etre D. Regardez


aussi M3 : i^2 ne depasse jamais i^2+10 — c'est la MARGE du budget qui la contient ;


l'Exercice 2 la retire.


**Pourquoi le budget ne tue pas l'argument.** Le point délicat — c'est là que 1965 innove
par rapport à Cantor et Turing — est que simuler $M_i$ pendant $t$ pas **coûte** quelque chose
à la machine qui diagonalise (à une machine universelle multitête : $O(t \log t)$, à cause du
coût de gestion des rubans et de la table de transitions). Si le budget $f$ est trop petit
devant le coût de la simulation, la machine $D$ elle-même explose et la diagonalisation
s'effondre. La parade exige deux ingrédients, tous deux dans le papier :

- une fonction $f$ **temps-constructible** : une machine qui, sur entrée de longueur $n$,
  s'arrête en exactement $\Theta(f(n))$ pas — on sait *chronométrer* sans dépasser ;
- un budget diagonal avec de la marge : $f$ assez grande pour payer la simulation des
  machines qu'elle diagonalise.

C'est exactement la source du facteur $\log$ du théorème — pas une scorie : c'est le **prix
de la simulation**, payé à chaque étage de la hiérarchie.


### Exercice 2 — Diagonaliser avec un budget serré

La construction ci-dessus utilise $f(i) = i^2 + 10$, qui **domine** le coût $i^2$ de $M_3$.
On veut voir l'argument vaciller : budget serré $f(i) = 2i + 6$.

**Objectif** : reprendre la boucle avec ce budget, compter les lignes où le verdict simulé
est `None` (budget épuisé), et dire en une phrase ce que cela illustre : *le budget doit
dominer le coût des machines diagonalisées, sinon la diagonale ne dit plus rien de leurs
verdicts réels*.

- **Indice :** M3(i) coute i*i pas ; comparez a 2i+6 pour i = 0..4, ligne par ligne.
- **Etape 1 :** recompter la colonne 'None' avec `long BudgetFSerre(int i) => 2 * i + 6;`.
- **Etape 2 :** pour ces lignes, D(i) vaut True par convention — mais M3 rend quand meme
  un verdict reel si on la laissait finir. La difference D vs M3 est-elle
  encore une preuve que M3 ne decide pas D en 2i+6 pas ? (oui — pourquoi ?)


In [8]:
#nullable enable
// Exercice 2 : a completer
int? lignes_budget_epuise = null; // TODO etudiant : nombre de lignes None avec f(i) = 2i+6
if (lignes_budget_epuise is null)
    Console.WriteLine("Exercice a completer : diagonaliser avec f(i) = 2i+6 et compter les budgets epuises");
else
{
    long BudgetFSerre(int i) => 2 * i + 6;
    int attendu = 0;
    for (int i = 0; i < familles.Length; i++)
        if (Simule(familles[i], i, BudgetFSerre(i)).Verdict is null) attendu++;
    if (lignes_budget_epuise != attendu)
        throw new Exception($"Exercice 2 : {attendu} lignes None attendues, {lignes_budget_epuise} fourni");
    Console.WriteLine($"Exercice 2 vérifié : {attendu} lignes None sur {familles.Length * familles.Length} (5 machines x 5 indices), budget f(i) = 2i+6.");
}


Exercice a completer : diagonaliser avec f(i) = 2i+6 et compter les budgets epuises


## 4. Le théorème de 1965, énoncé et instrumenté

> **Théorème (Hartmanis & Stearns 1965, forme multitête).** Pour toute fonction $f$
> temps-constructible,
> $$\mathrm{TIME}(f) \subsetneq \mathrm{TIME}(f \log f)$$
> — il existe un langage décidable en $O(f \log f)$ pas mais par **aucune** machine
> multitête en $O(f)$ pas.

Trois lectures, toutes dans l'esprit du papier :

1. **La hiérarchie est infinie** : en itérant $f \mapsto f \log f$, on obtient une chaîne
   stricte infinie de classes — « plus de temps ⟹ strictement plus de problèmes résolubles »,
   l'énoncé fondateur, vérifié une infinité de fois ;
2. **La dureté existe** : pour chaque budget, il existe des problèmes **prouvablement** au-delà —
   la diagonale de la section 3 en est la preuve d'existence constructive ;
3. **La limite de l'outil** : la diagonalisation sépare des budgets *éloignés* (d'un facteur
   $\log$), elle ne dit rien de $\mathrm{TIME}(n^{1.1})$ contre $\mathrm{TIME}(n)$ — et
   elle ne résoudra pas P ≠ NP (relativisation, Baker–Gill–Solovay 1975). Un fondateur qui
   connaît les limites de son propre outil : c'est aussi ça, 1965.

Les deux ingrédients du théorème, instrumentés : une **horloge** temps-constructible (le coût
réellement compté d'une écriture d'indices en binaire, $\Theta(n \log n)$ par construction),
et le **prix de la simulation** — le facteur dont la somme fait le $\log$ du théorème.


In [9]:
#nullable enable
// Nombre de bits de l'ecriture binaire : 1 pour 0, log2(v)+1 sinon (comme bin(i)[2:] en Python).
int Bits(int v) => v == 0 ? 1 : 32 - System.Numerics.BitOperations.LeadingZeroCount((uint)v);

long Horloge(int n)
{
    // Horloge temps-constructible : pour chaque position i de l'entree (0..n-1),
    // ecrit l'indice i en binaire sur le ruban de travail, bit par bit.
    // Chaque ecriture de bit = 1 pas compte ; le total est Theta(n log n) PAR CONSTRUCTION
    // (somme des longueurs binaires des indices).
    long pas = 0;
    for (int i = 0; i < n; i++)
        for (int b = 0; b < Bits(i); b++) pas++;
    return pas;
}

Console.WriteLine($"{"n",5} | {"horloge(n)",10} | {"n*ceil(log2 n)",14}");
Console.WriteLine(new string('-', 38));
foreach (int n in new[] { 4, 8, 16, 32, 64, 128, 256 })
    Console.WriteLine($"{n,5} | {Horloge(n),10} | {(long)n * Bits(n - 1),14}");

Console.WriteLine();
Console.WriteLine("L'horloge est sous la borne n*ceil(log2 n), du meme ordre : Theta(n log n) exact.");
Console.WriteLine("Temps-constructible = on sait chronometrer ce cout SANS le depasser : c'est le");
Console.WriteLine("premier ingredient du theoreme.");
Console.WriteLine();
Console.WriteLine("--- Prix de la simulation : interpreter une machine pas a pas ---");

(long Pas, long Cout) InterpreterPasAPas(MachineTuring machine, string mot, long maxPas)
{
    // Comme Run(), mais via une couche d'interpretation qui journalise son propre cout :
    // chaque pas simule paie une recherche dans la table des transitions, O(log |delta|).
    var rubans = Enumerable.Range(0, machine.K).Select(i => new Ruban(i == 0 ? mot : "")).ToList();
    string etat = machine.Initial;
    long cout = 0;
    long pas = 0;
    long largeurRecherche = Bits(machine.Delta.Count);   // log |delta| : cout d'une recherche dans la table
    while (!machine.Acceptants.Contains(etat) && pas < maxPas)
    {
        var lus = rubans.Select(r => r.Lire()).ToArray();
        cout += largeurRecherche;
        string cle = MachineTuring.Cle(etat, lus);
        if (!machine.Delta.TryGetValue(cle, out var regle)) break;  // blocage : rejet — l'interpretation s'arrete aussi
        etat = regle.NouvelEtat;
        for (int i = 0; i < machine.K; i++)
        {
            rubans[i].Ecrire(regle.Actions[i].Ecrire);
            rubans[i].Deplacer(regle.Actions[i].Direction);
        }
        pas++;
    }
    return (pas, cout);
}

foreach (var (nom, mach) in new (string, MachineTuring)[] { ("increment (|delta|=6)", inc), ("palindrome (|delta|=22)", pal) })
{
    string mot = string.Concat(Enumerable.Repeat("0110", 8)); // palindrome : les deux machines y tournent jusqu'au bout
    var (p, c) = InterpreterPasAPas(mach, mot, 50_000);
    if (p == 0) throw new Exception($"interpreteur : aucune pas simule pour {nom}");
    Console.WriteLine($"{nom,26} : {p,5} pas simules, cout interprete {c,6} -> facteur x{(double)c / p:F2}");
}

Console.WriteLine();
Console.WriteLine("Le facteur depend de la TAILLE DE DESCRIPTION de la machine simulee (log |delta|),");
Console.WriteLine("pas de la longueur d'entree : c'est le prix de la simulation, et la source du");
Console.WriteLine("facteur log dans TIME(f) < TIME(f log f) — paye a chaque etage de la hierarchie.");


    n | horloge(n) | n*ceil(log2 n)


--------------------------------------


    4 |          6 |              8


    8 |         18 |             24


   16 |         50 |             64


   32 |        130 |            160


... (8 outputs left out)

   palindrome (|delta|=22) :   849 pas simules, cout interprete   4245 -> facteur x5,00


Le facteur depend de la TAILLE DE DESCRIPTION de la machine simulee (log |delta|),


pas de la longueur d'entree : c'est le prix de la simulation, et la source du


facteur log dans TIME(f) < TIME(f log f) — paye a chaque etage de la hierarchie.


### Exercice 3 — La machine qui ne peut pas suivre

Construire sur le simulateur multitête une machine qui décide le langage
$L = \{ww \mid w \in \{0,1\}^*\}$ (un mot doublé) en zigzag quadratique, mesurer ses
pas pour des entrées de $n = 8, 16, 32$, et comparer au palindrome de la section 1 : même
famille de coût ou non ?

**Objectif** : la table de transitions, la série de comptes, et une phrase d'interprétation.

- **Indice :** comme le palindrome (comparer premier/dernier, rayer, recommencer) mais la
  parite doit etre gardee : si une seule case NON-rayee reste a la fin, rejeter —
  le palindrome l'accepte (centre seul), le doublon ne le doit pas.
- **Etape 1 :** deriver la table de D (section 1) : etat 'centre' qui rejette au lieu d'accepter.
- **Etape 2 :** mesurer pas(n) et comparer les rapports pas(n)/pas(n/2) aux deux machines.


In [10]:
#nullable enable
// Exercice 3 : a completer
Dictionary<string, RegleTuring>? transitions_doublement = null; // TODO etudiant : table derivee de `pal` avec garde de parite
string etatInitialDoublement = "scan";   // TODO etudiant : si votre etat initial differe, ajustez cette ligne
string etatAcceptantDoublement = "fin";   // TODO etudiant : si votre etat acceptant differe, ajustez cette ligne
if (transitions_doublement is null)
    Console.WriteLine("Exercice a completer : machine de {ww}, comptes de pas et comparaison au palindrome");
else
{
    if (transitions_doublement.Count == 0) throw new Exception("Exercice 3 : la table de transitions est vide");
    var doublement = new MachineTuring(1, transitions_doublement, etatInitialDoublement, new[] { etatAcceptantDoublement });
    bool EstDouble(string mot) => mot.Length % 2 == 0 && mot.Substring(0, mot.Length / 2) == mot.Substring(mot.Length / 2);
    foreach (var (mot, attendu) in new[] { ("0101", true), ("0011", true), ("01100110", true), ("0110", false), ("01", false), ("01010", false) })
    {
        var (etat, _) = doublement.Run(mot, 500_000);
        if ((etat == etatAcceptantDoublement) != attendu)
            throw new Exception($"Exercice 3 : verdict incorrect sur {mot} (attendu {attendu})");
    }
    var comptes = new List<(int n, long pas)>();
    foreach (int n in new[] { 8, 16, 32 })
    {
        string moitie = new string(Enumerable.Range(0, n / 2).Select(i => (i * 7) % 2 == 0 ? '0' : '1').ToArray());
        string mot = moitie + moitie;
        doublement.Run(mot, 500_000);
        comptes.Add((mot.Length, doublement.Pas));
    }
    Console.WriteLine($"Exercice 3 vérifié : {string.Join(", ", comptes.Select(c => $"n={c.n}: {c.pas} pas"))}");
    Console.WriteLine("Comparez ces comptes à ceux du palindrome (cellule de la section 1) : même famille de coût ?");
}


Exercice a completer : machine de {ww}, comptes de pas et comparaison au palindrome



(12,10): warning CS8321: La fonction locale 'EstDouble' est déclarée, mais jamais utilisée



## 5. Le Stearns inattendu : Arrow et les jeux répétés

L'homme qui a fondé la théorie de la complexité a **commencé ailleurs**, et ces débuts sont
déjà dans le dépôt :

- **Arrow, 1959.** Le tout premier papier de Stearns — étudiant à Carleton College — portait
  sur le **paradoxe d'Arrow**, publié dans *The American Mathematical Monthly*. Le dépôt
  possède la série complète : [SocialChoice-01 — Arrow](../../GameTheory/SocialChoice/ChoixSocial-01-Arrow-Impossibility-Theorem-Python.ipynb)
  et le lake [`game_theory_lean`](../../GameTheory/game_theory_lean/) avec
  `SocialChoice/Arrow.lean`. Relier l'impossibilité d'agrégation au destin du futur
  fondateur de la complexité : deux façons de dire *« certaines choses sont structurellement
  hors d'atteinte »*.
- **Jeux répétés, 1964–1966.** Stearns a travaillé à l'*Arms Control and Disarmament Agency*
  sur les **jeux répétés à information incomplète** — la théorie des jeux comme outil de
  contrôle des armements. Ce travail devint un chapitre du livre d'Aumann et Maschler (1995,
  prix Lanchester), dont la page de titre porte *« with the collaboration of Richard E.
  Stearns »* ; il appelait l'épisode *« the story of how I almost won the Lanchester
  Prize »*. Le dépôt possède [`game_theory_lean/RepeatedGames`](../../GameTheory/game_theory_lean/RepeatedGames/)
  (lake complet) et l'écho coopératif dans
  [ICT-13 — Axelrod](../../IIT/ICT-Series/ICT-13-AxelrodStrategicMorphodynamics-Python.ipynb).

Ces deux fils (impossibilité d'agrégation, jeux répétés sous information incomplète) seront
approfondis dans les volets 2 et 3 de l'hommage — l'encart présent suffit à dire le lien.

## 6. Le gap Mathlib, mesuré honnêtement

Peut-on formaliser le théorème de 1965 en Lean 4 sur Mathlib ? Mesure firsthand (13/09/2026,
méthode de l'analyse discrepancy — lister nommément l'étage absent) :

**Présent** dans `Mathlib/Computability/` : `TuringMachine/` (rubans, configurations,
machines à piles, `ToPartrec`, `Computable`), `Halting.lean` (**l'indécidabilité — le
prototype diagonal le plus proche**), `Reduce.lean` (réductions many-one), `TuringDegree.lean`,
et la théorie des automates (DFA/NFA/regex/Myhill–Nerode/grammaires hors-contexte).

**Absent — le verdict** : aucun dossier `Mathlib/Computability/Complexity/`. Ni $\mathrm{TIME}(f)$,
ni machine universelle avec **overhead temporel**, ni comptage de pas borné, ni théorème de
hiérarchie. Les vœux correspondants figurent encore à l'état d'aspiration dans
`docs/1000.yaml` (théorèmes de hiérarchie temps/espace, théorème de Cook). Corroboration :
le fil Zulip mathlib (déc. 2021) — le cadre pour « complexity classes like P » n'existe pas ;
l'effort Coq pour Cook–Levin a coûté plus de 100 heures rien que pour la simulation
langage→machine.

**Conséquence pour cette série** : pas de lake `complexity_lean` aujourd'hui — poser les
définitions sans preuve violerait la discipline sorry=0 du dépôt. Ce notebook reste
**pédagogique**, son encart « gap » joue le rôle du jumeau formel en attendant : si Mathlib
gagne une couche Complexity, la formalisation du théorème de hiérarchie temps sera le grain
naturel de révision de cette section.


## 7. Conclusion

| Geste | Où | Ce qui a été mesuré |
|---|---|---|
| Compter les pas | Section 1 | Comptes exacts : increment $\Theta(n)$ contre palindrome zigzag $\Theta(n^2)$, machine vérifiée (accepte/rejette) |
| Séparer | Section 2 | Trois croissances chronométrées ; subset-sum : force brute vs DP, même verdict (contrôle `throw` à chaque ligne), écart d'un facteur exponentiel |
| Diagonaliser | Section 3 | Construction exécutée : $D$ diffère de chaque $M_i$ sur sa diagonale, budget épuisé inclus dans l'argument |
| Énoncer | Section 4 | Horloge temps-constructible (coût réel compté, $\Theta(n\log n)$) ; prix de la simulation mesuré sur deux machines (facteur $\log \lvert\Delta\rvert$) |

**Ce que 1965 a fondé.** Avant Hartmanis et Stearns : « décidable ou non ». Après : une
échelle infinie de budgets, des classes strictement emboîtées, et l'idée que la dureté d'un
problème est une propriété **mesurable** — le geste que ce notebook a rejoué cellule par
cellule. La limite de l'outil (relativisation) est aussi dans l'héritage : savoir ce que la
diagonalisation ne prouvera pas fait partie de la leçon.

**Hommage.** Ce notebook travaille le théorème d'un homme qui a passé sa vie à mesurer pourquoi
les choses sont dures — et qui, étudiant, avait commencé par prouver qu'aucune agrégation
parfaite n'existe. Sources : [CACM In Memoriam](https://cacm.acm.org/news/in-memoriam-richard-e-stearns-1936-2026/),
[Computational Complexity blog](https://blog.computationalcomplexity.org/2026/09/richard-stearns-1936-2026.html).

## Références

— Hartmanis, J., & Stearns, R. E. (1965). « On the computational complexity of algorithms ».
*Transactions of the American Mathematical Society* 117, 285–306.
— Stearns, R. E. (1959). « The problem of aggregation of preferences in a socialist community ».
*The American Mathematical Monthly* 66(6), 485–486. (premier papier)
— Aumann, R. J., & Maschler, M. (1995). *Repeated Games with Incomplete Information*.
MIT Press — « with the collaboration of Richard E. Stearns ».
— Baker, T., Gill, J., & Solovay, R. (1975). « Relativizations of the P =? NP question ».
*SIAM Journal on Computing* 4(4), 431–442.
— Spafford, E., & Garfinkel, S. (2026). « Richard E. Stearns (1936–2026) ». *CACM In Memoriam*.
— Gasarch, W. (2026). « Richard Stearns 1936–2026 ». *Computational Complexity blog*.


## À retenir

1. **La ressource n'est pas le chrono, c'est le compte de pas.** `increment('1011')` rend `1100` en 8 pas ; le palindrome coûte 6,90× l'incrément à $n = 8$ et **96,99×** à $n = 128$ (12 609 pas contre 130). Le compte est déterministe, le chrono ne l'est qu'à un facteur machine près : quand $n$ passe au quadruple, le chrono du tri varie ×3,4 et celui des paires ×17,3, alors que le compte double exactement pour $2^n$.
2. **La diagonale sous budget tient sur cinq lignes.** $D$ diffère de chaque $M_i$ sur sa diagonale : **les cinq lignes du tableau**. La ligne $i = 4$ est le cas clé — $M_4(4)$ coûte $4^4 = 256$ pas pour un budget de **26**, donc verdict `None`, et $D(4) = \text{True}$ **sans** que la simulation ait terminé.
3. **La marge du budget est ce qui rend l'argument valide.** $M_3$ s'arrête à $i^2$ pas, toujours sous le budget $i^2 + 10$ : c'est cette marge qui la contient dans la classe, et l'Exercice 2 la retire.
4. **L'horloge est temps-constructible.** `horloge(n)` reste sous $n \cdot \lceil \log_2 n \rceil$ (6 contre 8 à $n = 4$, 1 794 contre 2 048 à $n = 256$) : de l'ordre $\Theta(n \log n)$, le premier ingrédient du théorème — on sait chronométrer ce coût **sans** le dépasser.
5. **La simulation a un prix mesuré.** L'incrément ($|\Delta| = 6$) coûte 34 pas simulés pour un coût interprété de 102 (**×3,00**) ; le palindrome ($|\Delta| = 22$) coûte 849 pas pour 4 245 (**×5,00**). Le facteur dépend de la taille de description de la machine simulée, pas de la longueur d'entrée : c'est la source du facteur $\log$ dans $\mathrm{TIME}(f) \subsetneq \mathrm{TIME}(f \log f)$.


## Vérifiez votre compréhension

1. La section 1 compare l'incrément et le palindrome par un compte de pas exact. Pourquoi le carnet juge-t-il le compte plus fiable que le chrono ?
2. À la ligne $i = 4$, la simulation rend `None` (256 pas pour un budget de 26) et $D(4) = \text{True}$. Pourquoi ce `None` **confirme** la diagonale au lieu de l'infirmer ?
3. Le carnet dit de $M_3$ que $i^2$ ne dépasse jamais $i^2 + 10$. Que se passe-t-il si l'on retire cette marge (Exercice 2) ?
4. Le prix de la simulation vaut ×3,00 pour une machine de description 6 et ×5,00 pour une machine de description 22. Quel facteur, en fonction de $|\Delta|$, explique ces deux nombres ?

<details><summary>Réponses</summary>

1. Parce que le chrono dépend de la machine (caches, charge, fréquence effective) alors que le compte de pas ne dépend que de la machine simulée et de l'entrée. La section 2 le vérifie : quand $n$ passe au quadruple, le chrono du tri varie ×3,4 et celui des paires ×17,3, alors que le compte double exactement — l'écart est celui de la constante machine.
2. Parce que `None` signifie « la machine ne termine pas dans le budget », donc qu'elle **n'appartient pas** à la classe de budget $f$ : elle ne peut donc pas être égale à $D(4)$. Si la simulation avait rendu `True` dans le budget, on aurait une collision ; c'est au contraire son échec dans le budget qui garantit que $D(4)$ peut valoir `True`.
3. Sans marge, une machine peut dépasser son budget tout en restant `None` (comme $M_4$) et la diagonale n'est plus définie de façon unique : on ne sait plus quelle valeur $D(i)$ doit prendre. L'Exercice 2 montre que la construction devient circulaire — il faut choisir les machines de façon que chacune tienne dans le budget de la précédente.
4. Le facteur croît comme $\log_2 |\Delta|$ : $\log_2 6 \approx 2{,}6$ et $\log_2 22 \approx 4{,}5$, contre 3,00 et 5,00 mesurés. C'est ce logarithme du facteur de description — et non la longueur de l'entrée — qui donne le facteur $\log$ dans l'énoncé du théorème.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Complexity-03c — Naviguer le Zoo exécutable](Complexity-03c-ComplexityZoo-Navigation-CSharp.ipynb) — les classes de la hiérarchie, cette fois jouées sur instances.
- **Dans la même série** : [Complexity-03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-CSharp.ipynb) — le mécanisme de diagonalisation avant d'être instrumenté ; [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — ce qu'une annonce d'ETH fait et ne fait pas.
- **Carnet voisin d'une autre série** : [Arrow — le théorème d'impossibilité](../../GameTheory/SocialChoice/ChoixSocial-01-Arrow-Impossibility-Theorem-Python.ipynb) — l'autre grande diagonale du dépôt : Stearns, avant la complexité, démontre qu'aucune agrégation parfaite n'existe.
- **Lectures** : la section « Références » de ce carnet pour l'appareil historique ; au gisement partagé, Baker, Gill et Solovay (1975) et Impagliazzo et Paturi (2001) sont archivés dans le dossier `Bibliographie IA/Automata/`.

> Carnet issu de l'EPIC #26534.
